# HafenCity NM5 Full Simulation

This notebook inspects the prepared HafenCity `nm5_full` inputs first, then submits a real COUP-noise task to the local NoiseModelling 5 service.

By default it uses a fast smoke-test package at `downloads/hafencity_full/citypyo/hafencity_half_10m`, which keeps the half AOI but downsamples DEM points to 10 m spacing. Change `CITY_PYO_USER` in the first code cell back to `hafencity_full` for the complete AOI. The notebook writes an override compose file under `outputs/` and bind-mounts `downloads/` into the Docker services so the large prepared data does not need to be copied into the image.

In [ ]:
from pathlib import Path
import base64
import csv
import html
import json
import os
import subprocess
import time
from collections import Counter
from urllib import error, request

ROOT = Path.cwd()
if not (ROOT / "docker-compose.yml").exists():
    ROOT = ROOT.parent

CITY_PYO_USER = "hafencity_half_10m"
PACKAGE = ROOT / "downloads" / "hafencity_full" / "citypyo" / CITY_PYO_USER
PROCESSED = ROOT / "downloads" / "hafencity_full" / "processed"
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

API_URL = "http://localhost:5001"
AUTH_USER = "dev"
AUTH_PASSWORD = "dev"
print("repo root:", ROOT)
print("local package:", PACKAGE)
print("package exists:", PACKAGE.exists())

## Display Helpers

The plotting helpers below use plain SVG/HTML so the notebook does not require `geopandas`, `folium`, or `matplotlib` in the host environment.

In [ ]:
try:
    from IPython.display import HTML, display
except Exception:
    HTML = None
    def display(value):
        print(value)

def load_json(path):
    with Path(path).open(encoding="utf-8") as handle:
        return json.load(handle)

def read_csv_rows(path):
    with Path(path).open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))

def iter_positions(coords):
    if not coords:
        return
    if isinstance(coords[0], (int, float)):
        yield float(coords[0]), float(coords[1])
        return
    for child in coords:
        yield from iter_positions(child)

def feature_collection_bbox(payload):
    xs, ys = [], []
    for feature in payload.get("features", []):
        geometry = feature.get("geometry") or {}
        for x, y in iter_positions(geometry.get("coordinates")):
            xs.append(x)
            ys.append(y)
    return (min(xs), min(ys), max(xs), max(ys)) if xs else None

def html_table(rows, max_rows=25):
    rows = list(rows)
    if not rows:
        return HTML("<p>No rows.</p>") if HTML else "No rows."
    columns = list(rows[0].keys())
    body = []
    body.append("<table style='border-collapse:collapse;font:13px sans-serif'>")
    body.append("<thead><tr>" + "".join(f"<th style='border:1px solid #ccc;padding:4px 6px;text-align:left'>{html.escape(str(c))}</th>" for c in columns) + "</tr></thead>")
    body.append("<tbody>")
    for row in rows[:max_rows]:
        body.append("<tr>" + "".join(f"<td style='border:1px solid #ddd;padding:4px 6px'>{html.escape(str(row.get(c, '')))}</td>" for c in columns) + "</tr>")
    body.append("</tbody></table>")
    if len(rows) > max_rows:
        body.append(f"<p>Showing {max_rows} of {len(rows)} rows.</p>")
    return HTML("\n".join(body)) if HTML else "\n".join(str(row) for row in rows[:max_rows])

def geometry_svg(geometry, sx, sy, style):
    geometry_type = geometry.get("type")
    coordinates = geometry.get("coordinates")
    parts = []
    if geometry_type == "Point":
        x, y = coordinates[:2]
        parts.append(f"<circle cx='{sx(x):.2f}' cy='{sy(y):.2f}' r='1.5' {style}/>")
    elif geometry_type == "MultiPoint":
        for x, y, *_ in coordinates:
            parts.append(f"<circle cx='{sx(x):.2f}' cy='{sy(y):.2f}' r='1.1' {style}/>")
    elif geometry_type == "LineString":
        points = " ".join(f"{sx(x):.2f},{sy(y):.2f}" for x, y, *_ in coordinates)
        parts.append(f"<polyline points='{points}' {style}/>")
    elif geometry_type == "MultiLineString":
        for line in coordinates:
            points = " ".join(f"{sx(x):.2f},{sy(y):.2f}" for x, y, *_ in line)
            parts.append(f"<polyline points='{points}' {style}/>")
    elif geometry_type == "Polygon":
        for ring_index, ring in enumerate(coordinates):
            points = " ".join(f"{sx(x):.2f},{sy(y):.2f}" for x, y, *_ in ring)
            parts.append(f"<polygon points='{points}' {style}/>")
    elif geometry_type == "MultiPolygon":
        for polygon in coordinates:
            for ring in polygon:
                points = " ".join(f"{sx(x):.2f},{sy(y):.2f}" for x, y, *_ in ring)
                parts.append(f"<polygon points='{points}' {style}/>")
    return "\n".join(parts)

def layer_style(feature, layer_name):
    props = feature.get("properties") or {}
    if layer_name == "roads":
        if props.get("road_type") == "railroad":
            return "fill='none' stroke='#7b2cbf' stroke-width='1.6' stroke-opacity='0.9'"
        color = "#d9480f" if props.get("traffic_count_source") else "#f08c00"
        return f"fill='none' stroke='{color}' stroke-width='1.2' stroke-opacity='0.85'"
    if layer_name == "ground":
        g = float(props.get("G", 0.5))
        if g <= 0.05:
            fill = "#748ffc"
        elif g < 0.3:
            fill = "#adb5bd"
        elif g < 0.8:
            fill = "#fcc419"
        else:
            fill = "#51cf66"
        return f"fill='{fill}' fill-opacity='0.45' stroke='{fill}' stroke-width='0.5' stroke-opacity='0.8'"
    if layer_name == "dem":
        return "fill='#1971c2' fill-opacity='0.35' stroke='none'"
    if layer_name == "result":
        colors = {1:'#2b8cbe',2:'#7bccc4',3:'#a8ddb5',4:'#ffffb2',5:'#fecc5c',6:'#fd8d3c',7:'#f03b20',8:'#bd0026'}
        color = colors.get(int(props.get("idiso", 0) or 0), '#777')
        return f"fill='{color}' fill-opacity='0.55' stroke='{color}' stroke-width='0.8' stroke-opacity='0.9'"
    if layer_name == "aoi":
        return "fill='#4dabf7' fill-opacity='0.12' stroke='#1864ab' stroke-width='2'"
    return "fill='#495057' fill-opacity='0.5' stroke='#212529' stroke-width='0.5'"

def show_geojson(payload, layer_name, title, max_features=None, bbox=None, width=900, height=520):
    features = payload.get("features", [])
    if max_features and len(features) > max_features:
        stride = max(1, len(features) // max_features)
        features = features[::stride][:max_features]
    draw_payload = {"type": "FeatureCollection", "features": features}
    bbox = bbox or feature_collection_bbox(draw_payload)
    if not bbox:
        display(HTML(f"<p>{html.escape(title)}: no geometry.</p>") if HTML else f"{title}: no geometry")
        return
    min_x, min_y, max_x, max_y = bbox
    pad_x = max((max_x - min_x) * 0.04, 1)
    pad_y = max((max_y - min_y) * 0.04, 1)
    min_x -= pad_x; max_x += pad_x; min_y -= pad_y; max_y += pad_y
    def sx(x):
        return 18 + (float(x) - min_x) / (max_x - min_x) * (width - 36)
    def sy(y):
        return height - 18 - (float(y) - min_y) / (max_y - min_y) * (height - 36)
    shapes = []
    for feature in features:
        shapes.append(geometry_svg(feature.get("geometry") or {}, sx, sy, layer_style(feature, layer_name)))
    svg = f"""
    <div style='font:14px sans-serif'>
      <h3 style='margin:0 0 6px'>{html.escape(title)}</h3>
      <div style='color:#555;margin-bottom:8px'>{len(features)} displayed feature(s), bbox EPSG:25832: {min_x:.1f}, {min_y:.1f}, {max_x:.1f}, {max_y:.1f}</div>
      <svg width='{width}' height='{height}' style='border:1px solid #ccc;background:#f8f9fa'>
        {''.join(shapes)}
      </svg>
    </div>
    """
    display(HTML(svg) if HTML else svg)


## Input Inventory

In [ ]:
layer_files = {
    "project_area": PACKAGE / "project_area.geojson",
    "buildings_upperfloor": PACKAGE / "upperfloor.geojson",
    "roads_and_rail": PACKAGE / "roads.geojson",
    "dem": PACKAGE / "dem.geojson",
    "ground_absorption": PACKAGE / "ground_absorption.geojson",
    "atmospheric_settings": PACKAGE / "atmospheric_settings.csv",
    "traffic_counts_reference": PACKAGE / "traffic_counts_reference.geojson",
    "alkis_reference": PACKAGE / "alkis_reference.geojson",
    "traffic_assignments": PROCESSED / "traffic_count_assignments_hafencity.csv",
    "gtfs_rail_assignments": PROCESSED / "gtfs_rail_assignments_hafencity.csv",
}

inventory = []
for name, path in layer_files.items():
    item = {"layer": name, "path": str(path.relative_to(ROOT)), "exists": path.exists(), "bytes": path.stat().st_size if path.exists() else 0}
    if path.exists() and path.suffix.lower() == ".geojson":
        item["features"] = len(load_json(path).get("features", []))
    elif path.exists() and path.suffix.lower() == ".csv":
        item["rows"] = len(read_csv_rows(path))
    inventory.append(item)

display(html_table(inventory, max_rows=30))

## Project Area

In [ ]:
project_area = load_json(PACKAGE / "project_area.geojson")
AOI_BBOX = feature_collection_bbox(project_area)
show_geojson(project_area, "aoi", "AOI / project_area.geojson")

## Buildings: LoD2-Derived `upperfloor.geojson`

In [ ]:
buildings = load_json(PACKAGE / "upperfloor.geojson")
heights = [float((feature.get("properties") or {}).get("HEIGHT", 0)) for feature in buildings.get("features", [])]
print("building features:", len(heights))
print("height min/max/avg:", min(heights), max(heights), round(sum(heights) / len(heights), 2))
show_geojson(buildings, "buildings", "LoD2 buildings / upperfloor.geojson", bbox=AOI_BBOX)

## Roads And Rail: OSM + Traffic Counts + GTFS

In [ ]:
roads = load_json(PACKAGE / "roads.geojson")
road_features = [feature for feature in roads.get("features", []) if (feature.get("properties") or {}).get("road_type") != "railroad"]
rail_features = [feature for feature in roads.get("features", []) if (feature.get("properties") or {}).get("road_type") == "railroad"]
traffic_assigned = [feature for feature in road_features if (feature.get("properties") or {}).get("traffic_count_source")]
gtfs_assigned = [feature for feature in rail_features if (feature.get("properties") or {}).get("gtfs_source")]
print("road features:", len(road_features))
print("rail features:", len(rail_features))
print("traffic-count-enriched roads:", len(traffic_assigned))
print("GTFS-enriched rail features:", len(gtfs_assigned))
print("road classes:", Counter((feature.get("properties") or {}).get("fclass") for feature in road_features))
print("rail classes:", Counter((feature.get("properties") or {}).get("fclass") for feature in rail_features))
show_geojson(roads, "roads", "Roads and rail / roads.geojson", bbox=AOI_BBOX)

## DEM Terrain Points

In [ ]:
dem = load_json(PACKAGE / "dem.geojson")
z_values = [float(feature["geometry"]["coordinates"][2]) for feature in dem.get("features", []) if len(feature.get("geometry", {}).get("coordinates", [])) >= 3]
print("DEM points:", len(z_values))
print("elevation min/max/avg:", min(z_values), max(z_values), round(sum(z_values) / len(z_values), 3))
show_geojson(dem, "dem", "DEM terrain points / dem.geojson (sampled)", max_features=6000, bbox=AOI_BBOX)

## Ground Absorption: ALKIS `G` Polygons

In [ ]:
ground = load_json(PACKAGE / "ground_absorption.geojson")
g_counter = Counter(float((feature.get("properties") or {}).get("G", 0)) for feature in ground.get("features", []))
print("ground polygons:", len(ground.get("features", [])))
print("G distribution:", dict(sorted(g_counter.items())))
show_geojson(ground, "ground", "ALKIS-derived ground absorption / ground_absorption.geojson", bbox=AOI_BBOX)

## Atmospheric Settings

In [ ]:
atmospheric_rows = read_csv_rows(PACKAGE / "atmospheric_settings.csv")
display(html_table(atmospheric_rows, max_rows=10))

## Traffic Count Assignments

In [ ]:
traffic_assignment_rows = read_csv_rows(PROCESSED / "traffic_count_assignments_hafencity.csv")
display(html_table(traffic_assignment_rows, max_rows=20))

## GTFS Rail Assignments

In [ ]:
gtfs_assignment_rows = read_csv_rows(PROCESSED / "gtfs_rail_assignments_hafencity.csv")
freqs = [float(row["trains_per_hour"]) for row in gtfs_assignment_rows]
print("GTFS assignment rows:", len(gtfs_assignment_rows))
print("trains/hour min/max/avg:", min(freqs), max(freqs), round(sum(freqs) / len(freqs), 2))
display(html_table(gtfs_assignment_rows, max_rows=12))

## Start The `nm5_full` Docker Stack

This cell writes a compose override file in `outputs/` and starts the API/worker services with `CITY_PYO=/app/downloads/hafencity_full/citypyo` and `NOISE_ENGINE=nm5_full`.

Set `START_STACK = False` if the stack is already running with the same settings.

In [ ]:
START_STACK = True
COMPOSE_OVERRIDE = OUTPUTS / "docker-compose.hafencity-nm5-full.yml"

override_yaml = f"""
services:
  api:
    environment:
      - CITY_PYO=/app/downloads/hafencity_full/citypyo
      - NOISE_ENGINE=nm5_full
      - CELERY_QUEUE=noise_nm5_full
      - CLIENT_ID=dev
      - CLIENT_PASSWORD=dev
      - REDIS_PASS=devredis
    volumes:
      - \"{(ROOT / 'downloads').resolve().as_posix()}:/app/downloads:ro\"
  worker_1:
    environment:
      - CITY_PYO=/app/downloads/hafencity_full/citypyo
      - NOISE_ENGINE=nm5_full
      - CELERY_QUEUE=noise_nm5_full
      - REDIS_PASS=devredis
    volumes:
      - \"{(ROOT / 'downloads').resolve().as_posix()}:/app/downloads:ro\"
  worker_2:
    environment:
      - CITY_PYO=/app/downloads/hafencity_full/citypyo
      - NOISE_ENGINE=nm5_full
      - CELERY_QUEUE=noise_nm5_full
      - REDIS_PASS=devredis
    volumes:
      - \"{(ROOT / 'downloads').resolve().as_posix()}:/app/downloads:ro\"
""".strip() + "\n"

COMPOSE_OVERRIDE.write_text(override_yaml, encoding="utf-8")
print("wrote", COMPOSE_OVERRIDE)

if START_STACK:
    subprocess.run([
        "docker", "compose",
        "-f", str(ROOT / "docker-compose.yml"),
        "-f", str(COMPOSE_OVERRIDE),
        "up", "-d", "--build",
    ], cwd=ROOT, check=True)
    print("nm5_full stack requested")

## Submit The Full Noise Simulation

The request below asks for GeoJSON contours. `traffic_quota=1.0` keeps the prepared traffic values. The API still requires `max_speed`; current COUP-noise scenario handling applies it only to adjustable road features.

In [ ]:
def auth_header(user, password):
    token = base64.b64encode(f"{user}:{password}".encode("utf-8")).decode("ascii")
    return {"Authorization": f"Basic {token}"}

def http_json(url, method="GET", payload=None, timeout=60):
    data = None
    headers = auth_header(AUTH_USER, AUTH_PASSWORD)
    if payload is not None:
        data = json.dumps(payload).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req = request.Request(url, data=data, headers=headers, method=method)
    try:
        with request.urlopen(req, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"HTTP {exc.code} for {url}: {body}") from exc

def wait_for_api(seconds=180):
    deadline = time.time() + seconds
    last_error = None
    while time.time() < deadline:
        try:
            http_json(f"{API_URL}/tasks/not-a-real-task", timeout=5)
            return
        except Exception as exc:
            last_error = exc
            time.sleep(3)
    raise RuntimeError(f"API did not become reachable: {last_error}")

def submit_task(payload):
    response = http_json(f"{API_URL}/task", method="POST", payload=payload, timeout=60)
    return response["taskId"]

def poll_task(task_id, poll_seconds=5, timeout_seconds=7200):
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        payload = http_json(f"{API_URL}/tasks/{task_id}", timeout=60)
        state = payload.get("taskState")
        print(time.strftime("%H:%M:%S"), state, "ready=", payload.get("resultReady"))
        if payload.get("resultReady"):
            if state == "FAILURE":
                raise RuntimeError(payload.get("result"))
            return payload["result"]
        time.sleep(poll_seconds)
    raise TimeoutError(f"Task {task_id} did not finish within {timeout_seconds} seconds")

TASK_PAYLOAD = {
    "city_pyo_user": CITY_PYO_USER,
    "result_format": "geojson",
    "noise_engine": "nm5_full",
    "max_speed": 50,
    "traffic_quota": 1.0,
    "wall_absorption": 0.69,
    "nm5_settings": {
        "receiver_height": 4,
        "max_cell_dist": 150,
        "road_width": 1.5,
        "max_area": 5000,
        "reflection_order": 0,
        "max_source_distance": 250,
        "max_reflection_distance": 50,
        "diff_vertical": False,
        "diff_horizontal": False,
        "iso_classes": "45,50,55,60,65,70,75,200",
    },
}

print("submitting CityPyo user:", TASK_PAYLOAD["city_pyo_user"])
print("submitting NM5 settings:")
print(json.dumps(TASK_PAYLOAD["nm5_settings"], indent=2))

wait_for_api()
task_id = submit_task(TASK_PAYLOAD)
print("submitted task:", task_id)
result_geojson = poll_task(task_id, poll_seconds=10, timeout_seconds=7200)

result_path = OUTPUTS / f"{CITY_PYO_USER}_nm5_full.geojson"
result_path.write_text(json.dumps(result_geojson, indent=2), encoding="utf-8")
print("saved", result_path)


## Result Contours

In [ ]:
if "result_geojson" not in globals():
    result_geojson = load_json(OUTPUTS / f"{CITY_PYO_USER}_nm5_full.geojson")

features = result_geojson.get("features", [])
print("result contour features:", len(features))
print("idiso distribution:", Counter((feature.get("properties") or {}).get("idiso") for feature in features))
show_geojson(result_geojson, "result", f"NM5 full result contours / {CITY_PYO_USER}_nm5_full.geojson", bbox=AOI_BBOX)

## Optional: Stop The Stack

Run this only when you are done inspecting results.

In [ ]:
STOP_STACK = False
if STOP_STACK:
    subprocess.run([
        "docker", "compose",
        "-f", str(ROOT / "docker-compose.yml"),
        "-f", str(COMPOSE_OVERRIDE),
        "down",
    ], cwd=ROOT, check=True)